# Amber RAG Comparison — No-RAG baseline (Ollama vs ChatGPT)

This notebook strips out retrieval entirely. Each query is answered by **both** LLMs with no context attached, so you can see what each model knows on its own.

Pair this with the two RAG notebooks to get a full 4-way comparison per question:

1. **Ollama + RAG** — `Sauce_Amber_RAG_Combined.ipynb`
2. **ChatGPT + RAG** — `Sauce_Amber_RAG_Combined_ChatGPT.ipynb`
3. **Ollama, no RAG** — this notebook
4. **ChatGPT, no RAG** — this notebook

The AmberRAG persona is kept, but the "use ONLY the provided Context" rules are removed since there is no context to ground on.

## 1) Setup

In [3]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

### OpenAI API key

Export your key before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
```

If `langchain-openai` is not installed yet:

```bash
pip install -U langchain-openai
```

Ollama also needs to be running locally (same as in the original notebook):

```bash
ollama serve  # in another terminal
ollama pull llama3.1:8b  # once
```

In [4]:
from dotenv import load_dotenv
load_dotenv()  # reads .env from the current folder into os.environ

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to your .env file."
    )
print("OPENAI_API_KEY detected (length={}).".format(len(os.environ["OPENAI_API_KEY"])))

OPENAI_API_KEY detected (length=164).


## 2) Two LLMs, same temperature
Temperature is 0 for both to make the comparison as deterministic as each backend allows.

In [5]:
llm_ollama  = ChatOllama(model="llama3.1:8b", temperature=0)
llm_openai  = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 3) No-RAG prompt
Same AmberRAG persona as the RAG notebooks, with the "use ONLY the provided Context" rules removed. The model is allowed to answer from its own training knowledge.

In [6]:
no_rag_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

CORE RULES-
1) Answer the question to the best of your knowledge about AMBER and AmberTools.
2) If you are not sure about a specific AMBER command, flag, filename, or parameter value,
   say so explicitly rather than guessing.
3) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
4) Do NOT mention any Persona, Identity, or Role in your answer.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance when appropriate.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Question: {question}

Answer:""")

## 4) Two chains — one per LLM, both no-RAG

In [7]:
chain_ollama = no_rag_prompt | llm_ollama | StrOutputParser()
chain_openai = no_rag_prompt | llm_openai | StrOutputParser()

## 5) Side-by-side helper
Runs the question through both LLMs and prints the two answers in a labeled, easy-to-compare layout.

In [8]:
def query_both(question: str):
    print("=" * 72)
    print("Question:", question.strip())
    print("=" * 72)

    print("\n--- Ollama (llama3.1:8b), NO RAG ---\n")
    try:
        print(chain_ollama.invoke(question))
    except Exception as e:
        print(f"[Ollama error] {e}")

    print("\n--- ChatGPT (gpt-4o-mini), NO RAG ---\n")
    try:
        print(chain_openai.invoke(question))
    except Exception as e:
        print(f"[OpenAI error] {e}")

    print()  # trailing newline

## 6) Run the same 8 queries used in the RAG notebooks

In [9]:
query_both('What is Amber?')

Question: What is Amber?

--- Ollama (llama3.1:8b), NO RAG ---

**Direct Answer:** AMBER (Assisted Model Building with Energy Refinement) is a molecular dynamics simulation software package used to study the behavior of molecules at the atomic level. It's a widely used tool in computational chemistry and biochemistry for simulating biomolecular systems, such as proteins, nucleic acids, and their interactions.

**Technical Explanation:** AMBER is based on classical mechanics and uses numerical methods to solve the equations of motion for a system of atoms. The software package includes tools for preparing molecular structures, calculating energies, and performing simulations using various algorithms, including molecular dynamics (MD), Monte Carlo (MC) simulations, and free energy calculations.

**Practical Guidance:** To get started with AMBER, you'll need to prepare your molecular structure in the PDB format and create an input file that specifies the simulation parameters. You can the

In [10]:
query_both('Why should SHAKE be disabled during minimization in AMBER?')

Question: Why should SHAKE be disabled during minimization in AMBER?

--- Ollama (llama3.1:8b), NO RAG ---

**Direct Answer:** SHAKE should be disabled during minimization in AMBER because it can lead to inaccurate energy calculations and convergence issues.

**Technical Explanation:** SHAKE is a constraint used to fix the bond lengths of hydrogen atoms, which can help improve the stability of simulations. However, during minimization, the goal is to optimize the positions of all atoms, including hydrogens, without any constraints. Enabling SHAKE during minimization can cause the energy calculation to be inaccurate because it artificially fixes the bond lengths, preventing the minimizer from properly optimizing the system.

**Practical Guidance:** To disable SHAKE during minimization in AMBER, use the `:BONDS` directive with the `SHAKE=off` flag. For example:
```
&cntrl
  imin = 1,
  ntx = 1,
  irest = 0,
  ntb = 2,
  cutnb = 8.0,
  cutnc = 7.0,
  iwrap = 1,
  :BONDS
    SHAKE = off
/


In [11]:
query_both('How do I obtain a Z-DNA structure from NAB?')

Question: How do I obtain a Z-DNA structure from NAB?

--- Ollama (llama3.1:8b), NO RAG ---

To obtain a Z-DNA structure from NAB (Nucleic Acid Builder), you can follow these steps:

1. Run the `sander` command with the `-ZDNA` flag to generate a Z-DNA structure.

Clear, direct answer.

The `-ZDNA` flag is used in conjunction with the `sander` command to generate a Z-DNA structure from an input NAB file. This flag tells AMBER to perform the necessary calculations to convert the DNA structure into its Z-DNA form.

Practical guidance: Make sure you have the latest version of AMBER installed and that your NAB file is properly formatted for Z-DNA conversion. You can check the AMBER documentation for more information on using the `-ZDNA` flag.

Technical explanation: The `sander` command in AMBER uses a variety of algorithms to perform molecular dynamics simulations, including those required for DNA structure conversion. The `-ZDNA` flag specifically enables the Z-DNA conversion algorithm, 

In [12]:
query_both('How can I get SHAKE to consider two different residue names to be water?')

Question: How can I get SHAKE to consider two different residue names to be water?

--- Ollama (llama3.1:8b), NO RAG ---

To make SHAKE consider two different residue names as water, you need to specify them in the `SHAKE` block of your input file. 

You should add a line with the following format:
```
SHAKE
  ...
  WATER RESIDUES = <residue1> <residue2>
  ...
```
Replace `<residue1>` and `<residue2>` with the actual residue names you want to consider as water.

For example, if you have two residues named `HOH` and `WAT`, your input file should contain:
```
SHAKE
  ...
  WATER RESIDUES = HOH WAT
  ...
```
This will instruct SHAKE to apply its constraints to these specified residue names, treating them as water molecules.

Technically, the `WATER RESIDUES` parameter is used in the SHAKE algorithm to identify which residues should be treated as water and have their bonds constrained. By specifying the desired residue names here, you can customize the behavior of SHAKE for your specific s

In [13]:
query_both('Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?')

Question: Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?

--- Ollama (llama3.1:8b), NO RAG ---

**Direct Answer**
Ubuntu is a recommended operating system (OS) for installing AMBER with CUDA support, including the NVIDIA GeForce RTX 3070. Specifically, Ubuntu 20.04 LTS or later versions are known to work well.

**Technical Explanation**
AMBER and AmberTools rely on various dependencies, such as OpenMPI, FFTW, and CUDA libraries, which have been extensively tested and optimized for Ubuntu. The Ubuntu package manager (apt) makes it easier to install and manage these dependencies. Additionally, the CUDA toolkit is officially supported by NVIDIA on Ubuntu.

**Practical Guidance**
To set up AMBER with CUDA on an RTX 3070 server using Ubuntu:

1. Install Ubuntu 20.04 LTS or later.
2. Update the package list: `sudo apt update`
3. Install necessary dependencies:
	* OpenMPI: `sudo apt install openmpi-bin libopenmpi-d

In [14]:
query_both('How do I use paramfit to generate force field parameters for boron-containing compounds?')

Question: How do I use paramfit to generate force field parameters for boron-containing compounds?

--- Ollama (llama3.1:8b), NO RAG ---

To use `paramfit` to generate force field parameters for boron-containing compounds, you need to prepare a suitable input file. 

**Direct Answer:** Run the following command in your terminal/command prompt:
```bash
paramfit -i input.inp -o output.prm
```
Replace `input.inp` with your prepared input file and `output.prm` with the desired name for the generated parameter file.

**Technical Explanation:**

1.  Prepare an input file (`input.inp`) that contains information about the boron-containing compound, including its molecular structure, atom types, bond orders, and any relevant constraints or restraints.
2.  The `paramfit` tool uses this input to generate a new parameter file (`output.prm`) based on the AMBER ff14SB force field.
3.  If you need custom parameters for specific atoms or bonds in your compound, you can specify them in the input file u

In [15]:
query_both('I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?')

Question: I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?

--- Ollama (llama3.1:8b), NO RAG ---

To extract the PDB files for the two states, you can use the `ptraj` tool from AmberTools.

**Direct Answer:** Run the following command in your terminal/command prompt:
```bash
ptraj -i ptraj.in -o stateA.pdb -o stateB.pdb
```
**Technical Explanation:**

In this command:

* `-i ptraj.in` specifies the input file for `ptraj`, which contains the trajectory analysis instructions.
* `-o stateA.pdb` and `-o stateB.pdb` specify the output files, where `stateA.pdb` will contain the PDB coordinates for State A (bonded disulphide bridge) and `stateB.pdb` will contain the PDB coordinates for State B (unbonded bridge).

**Practical Guidance:**

1. Create a new file 

In [16]:
query_both("""
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?
""")

Question: I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?

--- Ollama (llama3.1:8b), NO RAG ---

To identify parallel strands of DNA using nastruct, you can use the `bptype` flag with a value of 1 for parallel or 2 for anti-parallel.

The correct command should be:
```
guessbp bptype 1
```
This will instruct nastruct to guess the base pairing type and identify the molecule as parallel strands.

If you write "guessbp" alone, it will default to guessing the base pairing type without specifying the strand orientation. If you use "guessbp bptype para", it is not a valid command and may cause 